# Phase 3 — Content-Based Filtering

## Purpose of Content-Based Filtering
Content-Based Filtering (CBF) recommends new items to a user by matching the characteristics of the items they previously liked with the characteristics of other items in the database. In the context of our Travel Recommender System, we build a preference profile for each user based on the attributes (category, level, province, city, and clean description spots) of the tourist attractions they have rated highly (rating >= 4.0). We then recommend other attractions that have high similarity to this user profile.

## Brief Explanation of TF-IDF + Cosine Similarity
1. **TF-IDF (Term Frequency-Inverse Document Frequency)**: We construct a single text document ('content' column) for each attraction by combining its metadata attributes. TF-IDF conversion transforms this textual representation into a sparse numerical vector. Term Frequency (TF) measures how often a word appears in a specific attraction's description, while Inverse Document Frequency (IDF) dampens the weight of common words across all attractions (e.g., "attraction", "tourism") and increases the weight of rare, highly-specific words.
2. **Cosine Similarity**: Once the attractions are represented in the TF-IDF vector space, we calculate the cosine of the angle between the user's preference vector (which is the rating-weighted average of their highly-rated attractions' vectors) and each candidate attraction's vector. Cosine similarity measures orientation rather than magnitude, which is ideal for text/categorical vectors as it is invariant to document length.

## Import Section

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display

# Add project root directory to sys.path to import src
PROJECT_DIR = Path.cwd().parent
sys.path.append(str(PROJECT_DIR))

from src.preprocessing import load_dataset, prepare_attractions, prepare_interactions, train_test_split_by_user
from src.content_based import build_content_column, build_tfidf_matrix, recommend_attractions

## Load Dataset

In [ ]:
# Load raw dataset using the pre-implemented preprocessing pipeline
csv_path = PROJECT_DIR / "data" / "tourism_recommendation_dataset_en.csv"
print(f"Loading dataset from: {csv_path}")

dataset = load_dataset(str(csv_path))

# Prepare attractions and interactions
attraction_df = prepare_attractions(dataset)
interactions_df = prepare_interactions(dataset)

# Perform stratified train/test split by user
train_df, test_df = train_test_split_by_user(interactions_df, test_ratio=0.2, min_interactions=5, random_state=42)

print(f"Unique attractions prepared: {len(attraction_df)}")
print(f"Total interactions prepared: {len(interactions_df)}")
print(f"Train split interactions: {len(train_df)}")
print(f"Test split interactions: {len(test_df)}")

## Build Model

In [ ]:
# Build content representation by concatenating and weighting metadata features
cbf_df = build_content_column(attraction_df)

# Fit TF-IDF Vectorizer and create sparse attraction feature matrix
vectorizer, tfidf_matrix, attraction_index = build_tfidf_matrix(cbf_df)

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")
print(f"Number of terms in vocabulary: {len(vectorizer.vocabulary_)}")

## Generate Recommendations

In [ ]:
# Select a sample tourist_id with interactions in the training split
sample_tourist_id = 149

print(f"Generating recommendations for tourist_id: {sample_tourist_id}")
recommendations = recommend_attractions(
    tourist_id=sample_tourist_id,
    interactions_df=train_df,
    attraction_df=attraction_df,
    tfidf_matrix=tfidf_matrix,
    attraction_index=attraction_index,
    top_n=10
)

# Rename attraction_category to category and select columns to display as requested
display_cols = ["attraction_uid", "attraction_name", "category", "city", "similarity_score", "rank"]
recommendations_display = recommendations.rename(columns={"attraction_category": "category"})

if not recommendations_display.empty:
    display(recommendations_display[display_cols])
else:
    print("No recommendations generated.")

## Demonstrate Cold-Start Behavior

In [ ]:
# Demonstrate cold-start behavior with a user who has no interactions in the training set
cold_start_tourist_id = 99999

print(f"Generating recommendations for cold-start tourist_id: {cold_start_tourist_id}")
cold_recommendations = recommend_attractions(
    tourist_id=cold_start_tourist_id,
    interactions_df=train_df,
    attraction_df=attraction_df,
    tfidf_matrix=tfidf_matrix,
    attraction_index=attraction_index,
    top_n=10
)

# Display result
cold_recommendations_display = cold_recommendations.rename(columns={"attraction_category": "category"})
print(f"Returned DataFrame shape: {cold_recommendations_display.shape}")
display(cold_recommendations_display)

## Discussion

### Recommendations Generated Successfully
The content-based model successfully constructed a preference profile for the target user `149` using their highly-rated attractions in the training set (ratings >= 4.0). Using this profile, it calculated cosine similarities against all attraction TF-IDF vectors, yielding the top-10 most similar attractions. The results show high similarity scores for recommended attractions, reflecting strong semantic matching.

### Visited Attractions Excluded
An essential requirement of a recommendation system is that it should not recommend items the user has already visited/experienced. In the `recommend_attractions()` function, any `attraction_uid` present in the user's training interactions (regardless of rating) is excluded from the final recommendation candidate list. This ensures the output is comprised entirely of novel recommendations.

### Cold-Start Behaviour
The "cold-start" problem occurs when a new user enters the system with zero historical interactions or only low ratings (ratings < 4.0). Because the model relies on user-rated items to construct the preference profile, it cannot build a profile vector in such scenarios. Under the hood, `build_user_profile()` returns `None` for user `99999`, and `recommend_attractions()` safely handles this by returning an empty DataFrame with the correct schema, avoiding runtime errors.

### Why TF-IDF and Cosine Similarity are Appropriate
1. **TF-IDF**: Allows us to highlight distinctive attributes of attractions (like specific categories, levels, or signature tourist spots) while automatically discounting common words that do not differentiate attractions (e.g., generic descriptors). Additionally, by duplicating critical fields (such as `attraction_category` and `attraction_level`) during content column generation, we can explicitly bias the TF-IDF representation toward these vital features.
2. **Cosine Similarity**: Normalizes the length of the documents (or attraction metadata strings). It measures the angle between the user preference profile and attraction vectors in the high-dimensional term space, ensuring that longer descriptions do not artificially skew the similarity score compared to shorter, concise metadata.

### Q&A
Did the content-based recommendation successfully recommend relevant attractions to the user without duplicating visited ones?
Yes, for the sample `tourist_id = 149`, a list of 10 recommendations was generated. All visited attractions were successfully filtered out, and cold-start queries (such as for `tourist_id = 99999`) were safely handled by returning an empty DataFrame without raising any runtime errors.

### Data Analysis Key Findings
- Successfully built a user profile vector for user `149` based on highly-rated (rating >= 4.0) attraction features.
- Generated 10 unique, high-similarity recommendations for the target user, matching by attraction attributes such as categories, levels, and location characteristics.
- Safely identified cold-start cases and returned a structured, empty DataFrame containing the schema.

### Insights or Next Steps
- Proceed to implement and document the collaborative filtering model in Phase 4.
- Compare the content-based approach with collaborative filtering in Phase 5 to measure relative precision, recall, and coverage metrics.